In [1]:
import requests
import pandas as pd
import time
import re
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import asyncio
import pandas as pd
from googletrans import Translator

In [2]:
try:
    df = pd.read_csv('ru_cefr_short.csv')
    print("✅ Файл найден:")
except:
    print("❌ Файл не найден, используем тестовые данные:")
    test_data = {
        'fragment': [
            "Весной, летом и осенью почти каждую субботу он играет в футбол. У них в школе есть футбольная команда. А зимой он играет в хоккей. Ещё мы любим театр.",
            "Вчера я весь вечер повторял грамматику, учил слова. В контрольной работе я сделал 3 ошибки. Питер и Кен написали всё без ошибок. Преподаватель сказал, что они делают большие успехи.",
            "В такой ситуации особенно сложно работающим студентам, которые должны находить время и для учёбы, и для работы. Это нередко вызывает стресс.",
            "Как редкое животное они стоили очень дорого и являлись предметом роскоши. Археологи нашли останки этих животных в развалинах домов богатых людей четвёртого века нашей эры.",
            "Он увлёкся энтомологией ещё мальчиком и в детстве познакомился с основными учёными трудами в этой области.",
            "Так же радиация — это частица, которая летит на огромной скорости. Сама частица может быть почти любой: от атомного ядра до электрона или фотона."
        ],
        'textbook-assigned cefr level': [1, 2, 3, 4, 5, 6]
    }
    df = pd.DataFrame(test_data)

df

✅ Файл найден:


,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",1
2,На каждой двери красные плакаты и красные фона...,1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,1
...,...,...
7317,Утечка мозгов стала ключевым трендом междунаро...,6
7318,"По оценкам менеджеров «Промы», такая ситуация ...",6
7319,"Но это не мы, а техно-мемы заполоняют мир благ...",6
7320,Mapillary использует программное обеспечение д...,6


In [3]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['fragment'].values,
    df['textbook-assigned cefr level'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['textbook-assigned cefr level']
)

len(train_texts)

5857

In [4]:
texts_to_augment = []

for i in range(len(train_labels)):
  if train_labels[i] == 6:
    texts_to_augment.append(train_texts[i])


len(texts_to_augment)

120

In [5]:
async def main():
    print(f"Аугментируем {len(texts_to_augment)} текстов...")
    df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

    translator = Translator()

    translated_to = await translator.translate(texts_to_augment, src='ru', dest='uz')
    print("Переведено на узбекский")

    translated_to = await translator.translate([trans.text for trans in translated_to], src='uz', dest='eo')
    print("Переведено на эсперанто")
    
    translated_to = await translator.translate([trans.text for trans in translated_to], src='eo', dest='fr')
    print("Переведено на французский")

    translated_to = await translator.translate([trans.text for trans in translated_to], src='fr', dest='ja')
    print("Переведено на японский")

    translated_to = await translator.translate([trans.text for trans in translated_to], src='ja', dest='hu')
    print("Переведено на венгерский")

    translated_to = await translator.translate([trans.text for trans in translated_to], src='hu', dest='lt')
    print("Переведено на латышский")
    
    translated_to = await translator.translate([trans.text for trans in translated_to], src='lt', dest='be')
    print("Переведено на белорусский")

    translated_to = await translator.translate([trans.text for trans in translated_to], src='be', dest='pt')
    print("Переведено на португальский")

    translated_to = await translator.translate([trans.text for trans in translated_to], src='pt', dest='kk')
    print("Переведено на казахский")

    translated_to = await translator.translate([trans.text for trans in translated_to], src='kk', dest='sl')
    print("Переведено на словенский")
    
    translated_to = await translator.translate([trans.text for trans in translated_to], src='sl', dest='ru')
    print("Переведено обратно на русский")

    for n in range(len(texts_to_augment)):
        print(f"\n{n}. Реальный текст:\n\t{texts_to_augment[n]}")
        print(f"Аугметированный текст:\n\t{translated_to[n].text}")

        new_row = pd.DataFrame({
            'text': [texts_to_augment[n]],
            'augmented-text': [translated_to[n].text]
        })
        df_pred = pd.concat([df_pred, new_row], ignore_index=True)

    output_filename = "c2_augmented_nine_times_translated.csv"
    df_pred.to_csv(output_filename, index=False, encoding='utf-8-sig')


    return df_pred

df_pred = await main()

Аугментируем 120 текстов...
Переведено на узбекский
Переведено на эсперанто
Переведено на французский
Переведено на японский
Переведено на венгерский
Переведено на латышский
Переведено на белорусский
Переведено на португальский
Переведено на казахский
Переведено на словенский
Переведено обратно на русский

0. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	3:38-3:54 На самом деле радиация сама по себе не заразна. Например, высокие дозы радиации одного изотопа часто используются для лечения людей и стерилизации фруктов и овощей.

1. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Uber недавно объявил об инвестициях в $1